In [1]:
print("⏳ Đang kiểm tra và cài đặt các gói thư viện cần thiết...")
%pip install -q xgboost scikit-learn torch torch-geometric pandas numpy matplotlib scikit-learn

⏳ Đang kiểm tra và cài đặt các gói thư viện cần thiết...
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# === KHỞI TẠO HỆ THỐNG, ĐỒNG BỘ ĐƯỜNG DẪN & NẠP DỮ LIỆU LAI ===
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. Cấu hình hệ thống ghi nhật ký log đồng bộ với các file trước
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

device = torch.device('cpu')
logger.info(f"Hệ thống kích hoạt thiết bị xử lý an toàn: {str(device).upper()}")

# 3. THIẾT LẬP ĐƯỜNG DẪN: Đồng bộ với cấu trúc thư mục cục bộ của cậu
# Nếu notebook nằm chung thư mục gốc, Path(".") sẽ tự động định vị chính xác
PROJECT_DIR = Path(".") 
HYBRID_DIR = PROJECT_DIR / "data" / "gold" / "All_Beauty" / "hybrid_model"
logger.info(f"Đường dẫn kiểm tra gói dữ liệu lai: {HYBRID_DIR.resolve()}")

# ============================================
# 4. LOAD SEPARATE FEATURES FOR BASELINE vs HYBRID
# ============================================
logger.info("Đang nạp các feature sets riêng biệt...")

# BASELINE FEATURES: Text embeddings ONLY (để so sánh công bằng)
features_text_baseline = np.load(HYBRID_DIR / "text_embeddings_aligned.npy")  # (N, 384)
logger.info(f"✓ Baseline features (text only): {features_text_baseline.shape}")

# HYBRID FEATURES: Structural + Text (dùng cho GNN)
features_hybrid = np.load(HYBRID_DIR / "combined_features.npy")  # (N, struct_dim + 384)
logger.info(f"✓ Hybrid features (structural + text): {features_hybrid.shape}")

# Load labels and splits
labels = np.load(HYBRID_DIR / "labels.npy")
splits = np.load(HYBRID_DIR / "splits.npy")

# --- CHUẨN HÓA ĐẶC TRƯNG TĨNH ĐỂ CHỐNG SỤP ĐỔ MẠNG NEURAL ---
# For baseline (text only): normalize all dimensions
logger.info("Đang chuẩn hóa baseline features (text only)...")
scaler_baseline = StandardScaler()
features_text_baseline_norm = scaler_baseline.fit_transform(features_text_baseline)
logger.info(f"✓ Baseline features normalized: {features_text_baseline_norm.shape}")

# For hybrid features: normalize structural part only
total_dim = features_hybrid.shape[1]
text_dim = 384
struct_dim = total_dim - text_dim

logger.info(f"Đang chuẩn hóa hybrid features (structural part {struct_dim} dims)...")
scaler_hybrid = StandardScaler()
features_hybrid_norm = features_hybrid.copy()
features_hybrid_norm[:, :struct_dim] = scaler_hybrid.fit_transform(features_hybrid[:, :struct_dim])
logger.info(f"✓ Hybrid features normalized: {features_hybrid_norm.shape}")

# Phân tách tập chỉ số theo phân đoạn train/val/test
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

logger.info(f"✓ Baseline features (Node Features): {features_text_baseline_norm.shape}")
logger.info(f"✓ Hybrid features (Node Features): {features_hybrid_norm.shape}")
logger.info(f"✓ Tập Huấn luyện (Train Set): {len(train_idx):,} mẫu ({len(train_idx)/len(labels)*100:.1f}%)")
logger.info(f"✓ Tập Xác thực (Validation Set): {len(val_idx):,} mẫu ({len(val_idx)/len(labels)*100:.1f}%)")
logger.info(f"✓ Tập Kiểm thử (Test Set): {len(test_idx):,} mẫu ({len(test_idx)/len(labels)*100:.1f}%)")
logger.info("\n🎯 KEY DIFFERENCE:")
logger.info(f"  - Baseline XGBoost: Uses TEXT ONLY ({features_text_baseline_norm.shape[1]} features)")
logger.info(f"  - Hybrid GraphSAGE: Uses STRUCTURAL + TEXT ({features_hybrid_norm.shape[1]} features) + Graph Edges")

/Users/mac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/mac/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-10 23:36:58,548 - INFO - Hệ thống kích hoạt thiết bị xử lý an toàn: CPU
2026-06-10 23:36:58,549 - INFO - Đường dẫn kiểm tra gói dữ liệu lai: /Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model
2026-06-10 23:36:58,549 - INFO - Đang nạp các feature sets riêng biệt...
2026-06-10 23:36:59,285 - INFO - ✓ Baseline features (text only): (644689, 384)
2026-06-10 23:37:00,009 - INFO - ✓ Hybrid features (structural + text): (644689, 420)
2026-06-10 23:37:

Baseline Machine Learning

In [3]:
# === CELL 2: HUẤN LUYỆN MÔ HÌNH MỐC BASELINE XGBOOST (TEXT-ONLY FEATURES) ===
from xgboost import XGBClassifier

# ⚠️ IMPORTANT: XGBoost uses TEXT EMBEDDINGS ONLY (true baseline)
X_train_baseline = features_text_baseline_norm[train_idx]
X_test_baseline = features_text_baseline_norm[test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

logger.info(f"\n🔴 BASELINE MODEL - XGBoost with TEXT-ONLY features")
logger.info(f"  X_train shape: {X_train_baseline.shape} (text embeddings only)")
logger.info(f"  X_test shape: {X_test_baseline.shape}")

# Tính trọng số phạt phạt mất cân bằng lớp (Lớp 1 chiếm ~76% trong tập kiểm thử của cậu)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
logger.info(f"Trọng số cân bằng nhãn: 1v{pos_weight:.2f}")

logger.info("🚀 Đang huấn luyện mô hình mốc XGBoost Classifier...")
xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=pos_weight,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)
xgb_baseline.fit(X_train_baseline, y_train)
logger.info("✓ Hoàn tất huấn luyện mô hình mốc!")

xgb_preds = xgb_baseline.predict(X_test_baseline)
xgb_probs = xgb_baseline.predict_proba(X_test_baseline)[:, 1]

xgb_f1 = f1_score(y_test, xgb_preds, average='macro') # Dùng macro để cân bằng cả 2 lớp dữ liệu
xgb_auc = roc_auc_score(y_test, xgb_probs)

print("\n📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH MỐC BASELINE - XGBOOST (TEXT-ONLY)]:")
print(classification_report(y_test, xgb_preds, digits=4))
print(f"F1-Score: {xgb_f1:.4f} | AUC-ROC: {xgb_auc:.4f}")

2026-06-10 23:37:09,737 - INFO - 
🔴 BASELINE MODEL - XGBoost with TEXT-ONLY features
2026-06-10 23:37:09,738 - INFO -   X_train shape: (451282, 384) (text embeddings only)
2026-06-10 23:37:09,738 - INFO -   X_test shape: (96704, 384)
2026-06-10 23:37:09,739 - INFO - Trọng số cân bằng nhãn: 1v8.08
2026-06-10 23:37:09,740 - INFO - 🚀 Đang huấn luyện mô hình mốc XGBoost Classifier...
2026-06-10 23:37:27,153 - INFO - ✓ Hoàn tất huấn luyện mô hình mốc!



📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH MỐC BASELINE - XGBOOST (TEXT-ONLY)]:
              precision    recall  f1-score   support

           0     0.9689    0.9145    0.9409     87765
           1     0.4588    0.7118    0.5579      8939

    accuracy                         0.8957     96704
   macro avg     0.7138    0.8131    0.7494     96704
weighted avg     0.9217    0.8957    0.9055     96704

F1-Score: 0.7494 | AUC-ROC: 0.8790


PyTorch Geometric Data & Loader

In [4]:
# === CELL 3: DỰNG ĐỒ THỊ PYG TOÀN PHẦN TRÊN BỘ NHỚ RAM (HYBRID FEATURES) ===
from torch_geometric.data import Data

logger.info("\n🔵 HYBRID MODEL - GraphSAGE with STRUCTURAL + TEXT features + Graph Edges")
logger.info("Đang trích xuất liên kết cấu trúc đồ thị hành vi từ JSON...")
with open(HYBRID_DIR / "graph_structure.json", "r") as f:
    graph_json = json.load(f)

# Lấy danh sách cạnh đồng hành vi review-review (Cùng sản phẩm + cùng rating + cùng ngày/tháng)
review_edges = graph_json['edges_review_review']['edges']
edge_index = torch.tensor(review_edges, dtype=torch.long).t().contiguous()

# Đóng gói dữ liệu đồ thị lai - USE HYBRID FEATURES
x_tensor = torch.tensor(features_hybrid_norm, dtype=torch.float)
y_tensor = torch.tensor(labels, dtype=torch.long)
graph_data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

# Thiết lập mặt nạ Masks phân tách không gian nút
graph_data.train_mask = torch.tensor(splits == 0, dtype=torch.bool)
graph_data.val_mask = torch.tensor(splits == 1, dtype=torch.bool)
graph_data.test_mask = torch.tensor(splits == 2, dtype=torch.bool)

# Chuyển dịch toàn bộ cấu trúc sang thiết bị an toàn (CPU)
graph_data = graph_data.to(device)

logger.info(f"✓ Khởi tạo đồ thị PyG thành công: {graph_data.num_nodes:,} Nút | {graph_data.num_edges:,} Cạnh hành vi")
logger.info(f"✓ Feature input: {graph_data.num_features} dims (structural {struct_dim} + text {text_dim})")

2026-06-10 23:37:27,350 - INFO - 
🔵 HYBRID MODEL - GraphSAGE with STRUCTURAL + TEXT features + Graph Edges
2026-06-10 23:37:27,350 - INFO - Đang trích xuất liên kết cấu trúc đồ thị hành vi từ JSON...
2026-06-10 23:37:30,490 - INFO - ✓ Khởi tạo đồ thị PyG thành công: 644,689 Nút | 267,837 Cạnh hành vi
2026-06-10 23:37:30,498 - INFO - ✓ Feature input: 420 dims (structural 36 + text 384)


GraphSAGE Model Architecture

In [5]:
# === CELL 4: KIẾN TRÚC GRAPHSAGE LAI TỐI ƯU HÓA - CÓ RESIDUAL + BATCH NORM CHỐNG OVER-SMOOTHING ===
from torch_geometric.nn import SAGEConv
from torch.nn import BatchNorm1d

class GraphSAGEWithSkipEfficient(torch.nn.Module):
    """GraphSAGE cải tiến: Residual + Batch Normalization + 3 tầng chống over-smoothing"""
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        
        # Tầng 3 để tăng khả năng học
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn3 = BatchNorm1d(hidden_channels)
        
        # Classifier nhận đầu vào kết hợp (skip connection)
        self.classifier = nn.Linear(hidden_channels + in_channels, out_channels)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x_orig = x  # Lưu lại đặc trưng gốc để skip connection
        
        # 1. Tầng 1: SAGEConv + BatchNorm + ReLU + Dropout
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.4, training=self.training)
        
        # 2. Tầng 2: SAGEConv + BatchNorm + ReLU + Dropout
        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.4, training=self.training)
        
        # 3. Tầng 3: SAGEConv + BatchNorm + ReLU (tăng model capacity)
        h = self.conv3(h, edge_index)
        h = self.bn3(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        
        # Kết nối tắt: Nối h và x gốc
        return self.classifier(torch.cat([h, x_orig], dim=1))

# Khởi tạo mô hình cải tiến
gnn_model = GraphSAGEWithSkipEfficient(
    in_channels=graph_data.num_features, 
    hidden_channels=128, 
    out_channels=2
).to(device)

print("📐 CẤU TRÚC MẠNG GRAPHSAGE LAI CẢI TIẾN (3 tầng + BatchNorm + Residual):")
print(gnn_model)

📐 CẤU TRÚC MẠNG GRAPHSAGE LAI CẢI TIẾN (3 tầng + BatchNorm + Residual):
GraphSAGEWithSkipEfficient(
  (conv1): SAGEConv(420, 128, aggr=mean)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): SAGEConv(128, 128, aggr=mean)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): SAGEConv(128, 128, aggr=mean)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (classifier): Linear(in_features=548, out_features=2, bias=True)
)


Model Training & Validation Loop

In [8]:
# === CELL 5: HUẤN LUYỆN GRAPHSAGE CẢI TIẾN - LEARNING RATE SCHEDULE + EARLY STOPPING ===
class_weights = torch.tensor([1.0, float(pos_weight)], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.01, weight_decay=1e-4)

# Learning rate scheduler: Giảm dần khi val_loss không cải thiện
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

NUM_EPOCHS = 100
best_val_loss = float('inf')
patience_counter = 0
MAX_PATIENCE = 20

logger.info("🔥 Bắt đầu huấn luyện GraphSAGE cải tiến (3 layers + ReduceLROnPlateau)...")
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    gnn_model.train()
    optimizer.zero_grad()
    
    # Forward pass
    out = gnn_model(graph_data.x, graph_data.edge_index)
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    train_losses.append(loss.item())
    
    # Backward pass
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), max_norm=1.0)
    optimizer.step()
    
    # Validation
    gnn_model.eval()
    with torch.no_grad():
        val_out = gnn_model(graph_data.x, graph_data.edge_index)
        val_loss = criterion(val_out[graph_data.val_mask], graph_data.y[graph_data.val_mask])
    val_losses.append(val_loss.item())
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(gnn_model.state_dict(), '/tmp/best_gnn_model.pt')
    else:
        patience_counter += 1
    
    # Log theo dõi
    if epoch % 20 == 0 or epoch == 1:
        current_lr = optimizer.param_groups[0]['lr']
        logger.info(f"Epoch {epoch:03d}/{NUM_EPOCHS} -> Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | LR: {current_lr:.6f} | Patience: {patience_counter}/{MAX_PATIENCE}")
    
    # Dừng sớm nếu không cải thiện
    if patience_counter >= MAX_PATIENCE:
        logger.info(f"⛔ Early stopping at epoch {epoch} - Val loss không cải thiện trong {MAX_PATIENCE} epochs")
        break

# Load best model
gnn_model.load_state_dict(torch.load('/tmp/best_gnn_model.pt'))
logger.info("✅ Hoàn tất huấn luyện mô hình GNN cải tiến!")

2026-06-10 23:40:32,230 - INFO - 🔥 Bắt đầu huấn luyện GraphSAGE cải tiến (3 layers + ReduceLROnPlateau)...
2026-06-10 23:40:49,967 - INFO - Epoch 001/100 -> Train Loss: 0.1342 | Val Loss: 0.4733 | LR: 0.010000 | Patience: 0/20
2026-06-10 23:45:24,282 - INFO - Epoch 020/100 -> Train Loss: 0.0785 | Val Loss: 0.0736 | LR: 0.010000 | Patience: 2/20
2026-06-10 23:49:17,103 - INFO - Epoch 040/100 -> Train Loss: 0.0537 | Val Loss: 0.0539 | LR: 0.010000 | Patience: 8/20
2026-06-10 23:51:42,990 - INFO - ⛔ Early stopping at epoch 52 - Val loss không cải thiện trong 20 epochs
2026-06-10 23:51:43,139 - INFO - ✅ Hoàn tất huấn luyện mô hình GNN cải tiến!


Kiểm thử và Xuất bảng đối sánh thực nghiệm

In [9]:
# === CELL 6: KIỂM THỬ & PHÂN TÍCH CHI TIẾT + BẢNG ĐỐI SÁNH ===
gnn_model.eval()

logger.info("Đang thực hiện suy luận dự đoán trên tập kiểm thử độc lập...")
with torch.no_grad():
    out = gnn_model(graph_data.x, graph_data.edge_index)
    test_logits = out[graph_data.test_mask]
    
    gnn_probs = F.softmax(test_logits, dim=1)[:, 1].numpy()
    gnn_preds = test_logits.argmax(dim=1).numpy()
    true_labels = graph_data.y[graph_data.test_mask].numpy()

gnn_f1 = f1_score(true_labels, gnn_preds, average='macro')
gnn_auc = roc_auc_score(true_labels, gnn_probs)

print("\n📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH ĐỀ XUẤT GRAPHSAGE LAI + SEMANTICS]:")
print(classification_report(true_labels, gnn_preds, digits=4))

# === XÂY DỰNG BẢNG ĐỐI SÁNH ===
comparison_df = pd.DataFrame({
    'Mô hình': [
        '🔴 XGBoost (Baseline)',
        '🔵 GraphSAGE (Hybrid)'
    ],
    'Đặc trưng đầu vào': [
        'Text embeddings ONLY (384 dims)',
        'Structural + Text (420+ dims) + Graph Edges'
    ],
    'F1-Score (%)': [
        f"{xgb_f1*100:.2f}",
        f"{gnn_f1*100:.2f}"
    ],
    'AUC-ROC': [
        f"{xgb_auc:.4f}",
        f"{gnn_auc:.4f}"
    ],
    'Recall (Lớp 1)': [
        f"{classification_report(y_test, xgb_preds, output_dict=True)['1']['recall']*100:.2f}%",
        f"{classification_report(true_labels, gnn_preds, output_dict=True)['1']['recall']*100:.2f}%"
    ],
    'Ghi chú': [
        'Tree-based, không dùng graph info',
        'GNN + Graph structure + Behavior patterns'
    ]
})

print("\n📈 " + "="*140 + " BẢNG ĐỐI SÁNH CÔNG BẰNG (TEXT-ONLY vs HYBRID) " + "="*140)
print(comparison_df.to_string(index=False))
print("="*280)

print("\n💡 KEY FINDINGS:")
print(f"  1. Baseline (Text-only) F1: {xgb_f1:.4f}")
print(f"  2. Hybrid (Structural+Text+Graph) F1: {gnn_f1:.4f}")
if gnn_f1 > xgb_f1:
    improvement = (gnn_f1 - xgb_f1) / xgb_f1 * 100
    print(f"  3. Improvement: +{improvement:.2f}% (Hybrid tốt hơn vì khai thác cấu trúc mạng)")
else:
    improvement = (xgb_f1 - gnn_f1) / gnn_f1 * 100
    print(f"  3. XGBoost leads by {improvement:.2f}% (Cần optimize GraphSAGE hyperparameters)")

2026-06-10 23:52:05,357 - INFO - Đang thực hiện suy luận dự đoán trên tập kiểm thử độc lập...



📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH ĐỀ XUẤT GRAPHSAGE LAI + SEMANTICS]:
              precision    recall  f1-score   support

           0     0.9987    0.9849    0.9918     87765
           1     0.8696    0.9874    0.9248      8939

    accuracy                         0.9852     96704
   macro avg     0.9342    0.9861    0.9583     96704
weighted avg     0.9868    0.9852    0.9856     96704


📈 ============================================================================================================================================ BẢNG ĐỐI SÁNH CÔNG BẰNG (TEXT-ONLY vs HYBRID) ============================================================================================================================================
             Mô hình                           Đặc trưng đầu vào F1-Score (%) AUC-ROC Recall (Lớp 1)                                   Ghi chú
🔴 XGBoost (Baseline)             Text embeddings ONLY (384 dims)        74.94  0.8790         71.18%         Tree-based, không dùng gr